# Scaling Laws — Theory Notebook

Interactive implementations of scaling law mathematics for LLMs.

**Topics covered:**

1. **Power-law fitting** — fit L(N) = A·N^(-α) from data points
2. **Chinchilla loss formula** — L(N, D) = E + A/N^α + B/D^β
3. **Compute-optimal allocation** — Lagrange-derived N* and D* for given C
4. **IsoFLOP profiles** — find optimal N at fixed compute
5. **Kaplan vs Chinchilla comparison** — contrast the two scaling regimes
6. **Inference-optimal scaling** — lifecycle cost analysis
7. **Repeated data scaling** — multi-epoch degradation
8. **MoE effective parameters** — N_eff = N_active · E^η
9. **Test-time compute scaling** — best-of-N and chain-of-thought
10. **Scaling law extrapolation with uncertainty** — confidence intervals

In [ ]:
# === Setup ===
import numpy as np
np.random.seed(42)

try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

def print_table(headers, rows, col_widths=None):
    """Pretty-print a table with headers and rows."""
    if col_widths is None:
        col_widths = [max(len(str(h)), max(len(str(r[i])) for r in rows)) + 2
                      for i, h in enumerate(headers)]
    header_str = ''.join(str(h).ljust(w) for h, w in zip(headers, col_widths))
    print(header_str)
    print('-' * sum(col_widths))
    for row in rows:
        print(''.join(str(v).ljust(w) for v, w in zip(row, col_widths)))

def fmt_sci(x, digits=2):
    """Format number in scientific notation."""
    if x == 0:
        return '0'
    exp = int(np.floor(np.log10(abs(x))))
    mantissa = x / 10**exp
    if abs(exp) <= 2:
        return f'{x:.{digits}f}'
    return f'{mantissa:.{digits}f}×10^{exp}'

def fmt_params(n):
    """Format parameter count (e.g., 7.0B, 140M)."""
    if n >= 1e12:
        return f'{n/1e12:.1f}T'
    elif n >= 1e9:
        return f'{n/1e9:.1f}B'
    elif n >= 1e6:
        return f'{n/1e6:.0f}M'
    elif n >= 1e3:
        return f'{n/1e3:.0f}K'
    return str(int(n))

print("Setup complete ✓")

## §1 Power-Law Fitting

Given observed loss values at different parameter counts, fit the power law:

$$L(N) = A \cdot N^{-\alpha}$$

On log-log scale: $\log L = \log A - \alpha \log N$ → linear regression.

**Method**: take logarithm of both sides; fit slope ($-\alpha$) and intercept ($\log A$).

In [ ]:
# === §1: Power-Law Fitting ===
np.random.seed(42)

# --- 1a: Fit L(N) = A * N^(-alpha) from observations ---
# Observed data points (parameter count, loss)
N_obs = np.array([1e8, 3e8, 1e9, 3e9, 1e10, 3e10, 1e11])   # 100M to 100B
# Simulated loss values following L = 6.0 * N^(-0.076) + noise
A_true, alpha_true = 6.0, 0.076
L_obs = A_true * N_obs**(-alpha_true) + np.random.normal(0, 0.01, len(N_obs))

print("=== Power-Law Fitting ===\n")
print("Observed data:")
print(f"{'N':>12}  {'L':>8}")
print("-" * 22)
for n, l in zip(N_obs, L_obs):
    print(f"{fmt_params(n):>12}  {l:>8.4f}")

# Log-log linear regression: log(L) = log(A) - alpha * log(N)
log_N = np.log(N_obs)
log_L = np.log(L_obs)

# Fit: y = mx + b where y=log(L), x=log(N), m=-alpha, b=log(A)
# Using normal equations: [m, b] = (X^T X)^(-1) X^T y
X = np.column_stack([log_N, np.ones(len(log_N))])
coeffs = np.linalg.lstsq(X, log_L, rcond=None)[0]
alpha_fit = -coeffs[0]
A_fit = np.exp(coeffs[1])

print(f"\nFitted parameters:")
print(f"  A     = {A_fit:.4f}  (true: {A_true:.4f})")
print(f"  alpha = {alpha_fit:.4f}  (true: {alpha_true:.4f})")

# --- 1b: Extrapolate to N = 1T ---
N_extrap = np.array([1e11, 3e11, 1e12])
L_extrap = A_fit * N_extrap**(-alpha_fit)
print(f"\nExtrapolation:")
for n, l in zip(N_extrap, L_extrap):
    print(f"  L({fmt_params(n)}) = {l:.4f}")

# --- 1c: R² goodness of fit ---
L_pred = A_fit * N_obs**(-alpha_fit)
SS_res = np.sum((L_obs - L_pred)**2)
SS_tot = np.sum((L_obs - np.mean(L_obs))**2)
R2 = 1 - SS_res / SS_tot
print(f"\nGoodness of fit: R² = {R2:.6f}")

# --- 1d: Visualise ---
if HAS_MPL:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    
    # Log-log plot
    N_plot = np.logspace(7, 13, 200)
    L_fit_plot = A_fit * N_plot**(-alpha_fit)
    
    ax1.scatter(N_obs, L_obs, color='red', s=80, zorder=5, label='Observed')
    ax1.plot(N_plot, L_fit_plot, 'b-', linewidth=2, label=f'Fit: L = {A_fit:.2f}·N^(-{alpha_fit:.4f})')
    ax1.scatter(N_extrap, L_extrap, color='green', marker='*', s=150, zorder=5, label='Extrapolated')
    ax1.set_xscale('log')
    ax1.set_yscale('log')
    ax1.set_xlabel('Parameters N')
    ax1.set_ylabel('Loss L')
    ax1.set_title('Power Law Fit (log-log)')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Residuals
    residuals = L_obs - L_pred
    ax2.bar(range(len(residuals)), residuals, color='steelblue')
    ax2.axhline(y=0, color='black', linewidth=0.5)
    ax2.set_xlabel('Data Point Index')
    ax2.set_ylabel('Residual (L_obs - L_fit)')
    ax2.set_title('Fit Residuals')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('scaling_power_law_fit.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: scaling_power_law_fit.png")

## §2 Chinchilla Loss Formula

The Chinchilla loss model (Hoffmann et al. 2022):

$$L(N, D) = E + \frac{A}{N^\alpha} + \frac{B}{D^\beta}$$

Three terms:
- $E = 1.69$: irreducible entropy (floor)
- $A / N^\alpha$: parametric loss (model capacity)
- $B / D^\beta$: data loss (training data volume)

We implement this and compute loss for known models.

In [ ]:
# === §2: Chinchilla Loss Formula ===
np.random.seed(42)

# Chinchilla fitted parameters
CHINCHILLA = {
    'A': 406.4,
    'B': 410.7,
    'alpha': 0.34,
    'beta': 0.28,
    'E': 1.69
}

def chinchilla_loss(N, D, params=CHINCHILLA):
    """Compute loss using Chinchilla formula.
    
    L(N, D) = E + A/N^alpha + B/D^beta
    
    Args:
        N: number of parameters (non-embedding)
        D: number of training tokens
        params: dict with A, B, alpha, beta, E
    Returns:
        loss (nats)
    """
    p = params
    return p['E'] + p['A'] / N**p['alpha'] + p['B'] / D**p['beta']

# --- 2a: Loss decomposition for known models ---
print("=== Chinchilla Loss Formula ===\n")

models = [
    ('GPT-3',      175e9, 300e9),
    ('Gopher',     280e9, 300e9),
    ('Chinchilla', 70e9,  1.4e12),
    ('LLaMA-1 7B', 7e9,   1e12),
    ('LLaMA-3 8B', 8e9,   15e12),
    ('PaLM',       540e9, 780e9),
]

p = CHINCHILLA
print(f"{'Model':<16} {'N':>10} {'D':>10} {'L_param':>8} {'L_data':>8} {'L_total':>8} {'PPL':>8}")
print("-" * 76)
for name, N, D in models:
    L_param = p['A'] / N**p['alpha']
    L_data = p['B'] / D**p['beta']
    L_total = p['E'] + L_param + L_data
    ppl = np.exp(L_total)
    print(f"{name:<16} {fmt_params(N):>10} {fmt_params(D):>10} "
          f"{L_param:>8.4f} {L_data:>8.4f} {L_total:>8.4f} {ppl:>8.2f}")

# --- 2b: Which term dominates? ---
print("\n--- Dominance analysis ---")
for name, N, D in models:
    L_param = p['A'] / N**p['alpha']
    L_data = p['B'] / D**p['beta']
    ratio = L_data / L_param
    dominant = 'data-limited' if ratio > 1 else 'param-limited'
    print(f"  {name:<16} L_data/L_param = {ratio:.2f}  → {dominant}")

## §3 Compute-Optimal Allocation (Lagrange Derivation)

Minimise $L(N, D)$ subject to $C = 6ND$.

**Lagrange result:**

$$N^* = \left(\frac{A\alpha}{B\beta}\right)^{1/(\alpha+\beta)} \cdot \left(\frac{C}{6}\right)^{\beta/(\alpha+\beta)}$$

$$D^* = \left(\frac{B\beta}{A\alpha}\right)^{1/(\alpha+\beta)} \cdot \left(\frac{C}{6}\right)^{\alpha/(\alpha+\beta)}$$

We implement this and verify against known results.

In [ ]:
# === §3: Compute-Optimal Allocation ===
np.random.seed(42)

def chinchilla_optimal(C, params=CHINCHILLA):
    """Compute Chinchilla-optimal N* and D* for given compute budget.
    
    Derived from Lagrange multiplier optimisation of:
        min L(N, D) = E + A/N^alpha + B/D^beta
        s.t. 6ND = C
    
    Args:
        C: total compute budget (FLOPs)
        params: Chinchilla parameters dict
    Returns:
        N_opt, D_opt, L_opt
    """
    A, B = params['A'], params['B']
    alpha, beta = params['alpha'], params['beta']
    E = params['E']
    
    # Scaling exponents
    a = beta / (alpha + beta)      # N exponent
    b = alpha / (alpha + beta)     # D exponent
    
    # Coefficient
    G = (A * alpha / (B * beta))**(1 / (alpha + beta))
    
    # Optimal allocation
    N_opt = G * (C / 6)**a
    D_opt = (1 / G) * (C / 6)**b
    L_opt = chinchilla_loss(N_opt, D_opt, params)
    
    return N_opt, D_opt, L_opt

# --- 3a: Compute-optimal for different budgets ---
print("=== Compute-Optimal Allocation ===\n")

p = CHINCHILLA
a = p['beta'] / (p['alpha'] + p['beta'])
b = p['alpha'] / (p['alpha'] + p['beta'])
print(f"Scaling exponents:  a (N) = {a:.4f},  b (D) = {b:.4f}")
print(f"Sum a + b = {a+b:.4f} (should be 1.0)\n")

compute_budgets = [1e18, 1e19, 1e20, 1e21, 1e22, 1e23, 1e24, 1e25]

print(f"{'C (FLOPs)':>14} {'N*':>12} {'D*':>12} {'D*/N*':>8} {'L*':>8} {'PPL':>8}")
print("-" * 68)
for C in compute_budgets:
    N_opt, D_opt, L_opt = chinchilla_optimal(C)
    ratio = D_opt / N_opt
    ppl = np.exp(L_opt)
    print(f"{fmt_sci(C):>14} {fmt_params(N_opt):>12} {fmt_params(D_opt):>12} "
          f"{ratio:>8.1f} {L_opt:>8.4f} {ppl:>8.2f}")

# --- 3b: Verify C = 6*N*D ---
print("\n--- Verification: C = 6·N·D ---")
for C in [1e22, 1e23, 1e24]:
    N_opt, D_opt, _ = chinchilla_optimal(C)
    C_check = 6 * N_opt * D_opt
    print(f"  C = {fmt_sci(C)},  6·N*·D* = {fmt_sci(C_check)},  "
          f"match: {abs(C_check/C - 1) < 1e-6}")

# --- 3c: Optimal loss scaling with compute ---
gamma = p['alpha'] * p['beta'] / (p['alpha'] + p['beta'])
print(f"\nOptimal loss exponent γ = αβ/(α+β) = {gamma:.4f}")
print(f"Loss reduction per 10× compute: {10**(-gamma)*100:.1f}% of previous")
print(f"  (i.e., {(1-10**(-gamma))*100:.1f}% loss reduction per 10× compute)")

## §4 IsoFLOP Profiles

Fix compute $C$; vary $N$ (and $D = C/(6N)$); plot loss vs $N$.

The minimum of this U-shaped curve gives the optimal $N^*$ for that $C$.
Repeat for multiple $C$ values to map the compute-optimal frontier.

In [ ]:
# === §4: IsoFLOP Profiles ===
np.random.seed(42)

def isoflop_profile(C, N_range, params=CHINCHILLA):
    """Compute loss for a range of N values at fixed compute C.
    
    For each N, D = C/(6N). Returns array of losses.
    """
    losses = []
    for N in N_range:
        D = C / (6 * N)
        if D > 0:
            losses.append(chinchilla_loss(N, D, params))
        else:
            losses.append(np.inf)
    return np.array(losses)

# --- 4a: Single IsoFLOP profile at C = 10^21 ---
print("=== IsoFLOP Profiles ===\n")

C_ex = 1e21
N_range = np.logspace(7, 11, 50)  # 10M to 100B
losses = isoflop_profile(C_ex, N_range)

# Find minimum
idx_min = np.argmin(losses)
N_min = N_range[idx_min]
L_min = losses[idx_min]

# Compare with analytical optimal
N_opt_analytical, D_opt_analytical, L_opt_analytical = chinchilla_optimal(C_ex)

print(f"C = {fmt_sci(C_ex)} FLOPs")
print(f"  Grid search:  N* = {fmt_params(N_min)},  L* = {L_min:.4f}")
print(f"  Analytical:   N* = {fmt_params(N_opt_analytical)},  L* = {L_opt_analytical:.4f}")

# --- 4b: Tabulate some points on IsoFLOP curve ---
test_Ns = [1e8, 3e8, 1e9, 3e9, 1e10, 3e10]
print(f"\n{'N':>12} {'D = C/(6N)':>14} {'L(N,D)':>10} {'L_param':>10} {'L_data':>10}")
print("-" * 60)
p = CHINCHILLA
for N in test_Ns:
    D = C_ex / (6 * N)
    L = chinchilla_loss(N, D)
    L_p = p['A'] / N**p['alpha']
    L_d = p['B'] / D**p['beta']
    print(f"{fmt_params(N):>12} {fmt_params(D):>14} {L:>10.4f} {L_p:>10.4f} {L_d:>10.4f}")

# --- 4c: Multiple IsoFLOP profiles ---
if HAS_MPL:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    compute_levels = [1e19, 1e20, 1e21, 1e22, 1e23, 1e24]
    colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(compute_levels)))
    
    optima_N = []
    optima_L = []
    
    for C_val, color in zip(compute_levels, colors):
        N_r = np.logspace(6, 12, 200)
        L_r = isoflop_profile(C_val, N_r)
        
        # Only plot where loss is reasonable
        mask = L_r < 10
        ax.plot(N_r[mask], L_r[mask], color=color, linewidth=2,
                label=f'C = 10^{int(np.log10(C_val))}')
        
        # Mark optimum
        N_o, D_o, L_o = chinchilla_optimal(C_val)
        ax.scatter([N_o], [L_o], color=color, s=100, zorder=5,
                   edgecolors='black', linewidth=1)
        optima_N.append(N_o)
        optima_L.append(L_o)
    
    # Connect optima (compute-optimal frontier)
    ax.plot(optima_N, optima_L, 'k--', linewidth=2, alpha=0.5,
            label='Compute-optimal frontier')
    
    ax.set_xscale('log')
    ax.set_xlabel('Parameters N', fontsize=12)
    ax.set_ylabel('Loss L (nats)', fontsize=12)
    ax.set_title('IsoFLOP Profiles with Compute-Optimal Frontier', fontsize=14)
    ax.legend(fontsize=9)
    ax.set_ylim(1.6, 5.0)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('scaling_isoflop_profiles.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: scaling_isoflop_profiles.png")

## §5 Kaplan vs Chinchilla Comparison

Kaplan (2020): $N^* \propto C^{0.73}$, $D^* \propto C^{0.27}$  
Chinchilla (2022): $N^* \propto C^{0.50}$, $D^* \propto C^{0.50}$

We compare the optimal allocations and resulting losses.

In [ ]:
# === §5: Kaplan vs Chinchilla Comparison ===
np.random.seed(42)

# Kaplan allocation (approximate power-law fit to their results)
def kaplan_optimal(C):
    """Kaplan-style allocation: N ∝ C^0.73, D ∝ C^0.27.
    
    Normalised so that C = 6*N*D holds.
    """
    # N ∝ C^0.73 and D = C/(6N)
    # Calibrate coefficient so D/N ~ 1.7 at C = 10^24 (GPT-3 regime)
    k_n = 0.0085  # calibration constant
    N_opt = k_n * C**0.73
    D_opt = C / (6 * N_opt)
    L_opt = chinchilla_loss(N_opt, D_opt)  # evaluate using Chinchilla formula
    return N_opt, D_opt, L_opt

print("=== Kaplan vs Chinchilla ===\n")

compute_budgets = [1e20, 1e21, 1e22, 1e23, 1e24]

print(f"{'C':>12} | {'Kaplan N*':>12} {'Kaplan D*':>12} {'L_K':>8} | "
      f"{'Chinch N*':>12} {'Chinch D*':>12} {'L_C':>8} | {'ΔL':>7}")
print("-" * 105)

for C in compute_budgets:
    N_k, D_k, L_k = kaplan_optimal(C)
    N_c, D_c, L_c = chinchilla_optimal(C)
    delta = L_k - L_c
    print(f"{fmt_sci(C):>12} | {fmt_params(N_k):>12} {fmt_params(D_k):>12} {L_k:>8.4f} | "
          f"{fmt_params(N_c):>12} {fmt_params(D_c):>12} {L_c:>8.4f} | {delta:>+7.4f}")

print("\n  ΔL > 0 means Kaplan wastes compute (loss higher than Chinchilla-optimal)")

# --- 5b: Under-training analysis ---
print("\n--- Under-Training Analysis for Known Models ---")
models_ut = [
    ('GPT-3',  175e9, 300e9),
    ('Gopher', 280e9, 300e9),
    ('PaLM',   540e9, 780e9),
    ('MT-NLG', 530e9, 270e9),
]

print(f"{'Model':<12} {'N':>10} {'D_actual':>12} {'D_optimal':>12} {'Ratio':>8} {'L_actual':>10} {'L_optimal':>10}")
print("-" * 82)
for name, N, D_actual in models_ut:
    D_optimal = 20 * N  # Chinchilla rule of thumb
    ratio = D_optimal / D_actual
    L_actual = chinchilla_loss(N, D_actual)
    L_optimal = chinchilla_loss(N, D_optimal)
    print(f"{name:<12} {fmt_params(N):>10} {fmt_params(D_actual):>12} {fmt_params(D_optimal):>12} "
          f"{ratio:>8.1f}× {L_actual:>10.4f} {L_optimal:>10.4f}")

## §6 Inference-Optimal Scaling

Total lifecycle cost:

$$C_{\text{lifecycle}} = 6ND + 2N \cdot Q$$

where $Q = T_{\text{output}} \times n_{\text{queries}}$ is total inference tokens.

For high $Q$: smaller model trained on more data is optimal.

In [ ]:
# === §6: Inference-Optimal Scaling ===
np.random.seed(42)

def lifecycle_cost(N, D, Q):
    """Total lifecycle cost (training + inference).
    
    Args:
        N: parameters
        D: training tokens
        Q: total inference tokens (output_tokens × queries)
    Returns:
        Total FLOPs
    """
    C_train = 6 * N * D
    C_infer = 2 * N * Q  # forward pass only
    return C_train + C_infer

print("=== Inference-Optimal Scaling ===\n")

# --- 6a: Compare two models at same loss ---
# Model A: 70B params, 1.4T tokens (Chinchilla-optimal)
# Model B: 7B params, ~10T tokens (LLaMA-style overtrained)
N_A, D_A = 70e9, 1.4e12
N_B, D_B = 7e9, 10e12  # overtrained for inference

L_A = chinchilla_loss(N_A, D_A)
L_B = chinchilla_loss(N_B, D_B)

print(f"Model A (Chinchilla-opt): N = {fmt_params(N_A)}, D = {fmt_params(D_A)}, L = {L_A:.4f}")
print(f"Model B (Inference-opt):  N = {fmt_params(N_B)}, D = {fmt_params(D_B)}, L = {L_B:.4f}")
print(f"Loss difference: {L_A - L_B:.4f} nats")

# --- 6b: Cost crossover ---
C_train_A = 6 * N_A * D_A
C_train_B = 6 * N_B * D_B
print(f"\nTraining cost:")
print(f"  Model A: {fmt_sci(C_train_A)} FLOPs")
print(f"  Model B: {fmt_sci(C_train_B)} FLOPs")
print(f"  Model B costs {C_train_B/C_train_A:.1f}× more to train")

# Per-query inference cost ratio
infer_ratio = N_A / N_B
print(f"\nInference cost per query:")
print(f"  Model A: {infer_ratio:.0f}× more expensive than Model B")

# Crossover: when does Model B become cheaper overall?
# C_train_B + 2*N_B*Q = C_train_A + 2*N_A*Q
# Q_cross = (C_train_B - C_train_A) / (2*(N_A - N_B))
tokens_per_query = 500  # average response length
if C_train_B > C_train_A:
    Q_cross = (C_train_B - C_train_A) / (2 * (N_A - N_B))
    queries_cross = Q_cross / tokens_per_query
    print(f"\nCrossover point (Model B becomes cheaper):")
    print(f"  Q = {fmt_sci(Q_cross)} total inference tokens")
    print(f"  ≈ {fmt_sci(queries_cross)} queries (at {tokens_per_query} tokens/query)")

# --- 6c: Lifecycle cost at various query volumes ---
print(f"\n{'Queries':>14} {'Cost A':>16} {'Cost B':>16} {'Cheaper':>10}")
print("-" * 60)
for n_q in [1e6, 1e7, 1e8, 1e9, 1e10]:
    Q = n_q * tokens_per_query
    cost_A = lifecycle_cost(N_A, D_A, Q)
    cost_B = lifecycle_cost(N_B, D_B, Q)
    cheaper = 'B' if cost_B < cost_A else 'A'
    print(f"{fmt_sci(n_q):>14} {fmt_sci(cost_A):>16} {fmt_sci(cost_B):>16} {cheaper:>10}")

## §7 Repeated Data Scaling

Training with $R$ epochs on $D$ unique tokens:

$$L(N, D, R) \approx L(N, D \cdot R^{1-c}), \quad c \approx 0.28$$

Effective data scales sub-linearly with repetitions.

In [ ]:
# === §7: Repeated Data Scaling ===
np.random.seed(42)

def effective_data(D_unique, R, c=0.28):
    """Compute effective data from repeated tokens.
    
    D_eff = D_unique * R^(1-c)
    
    At R=1: D_eff = D_unique (baseline)
    At R=4: D_eff < 4 * D_unique (diminishing returns)
    """
    return D_unique * R**(1 - c)

def loss_repeated(N, D_unique, R, c=0.28, params=CHINCHILLA):
    """Loss with repeated data."""
    D_eff = effective_data(D_unique, R, c)
    return chinchilla_loss(N, D_eff, params)

print("=== Repeated Data Scaling ===\n")

# --- 7a: Effective data for different repetitions ---
D_unique = 1e12  # 1T unique tokens
N = 7e9  # 7B model

print(f"Model: {fmt_params(N)} params,  Unique data: {fmt_params(D_unique)} tokens\n")

print(f"{'Epochs (R)':>10} {'D_total':>12} {'D_eff':>12} {'Efficiency':>12} {'Loss':>8}")
print("-" * 58)
for R in [1, 2, 4, 8, 16, 32]:
    D_total = D_unique * R
    D_eff = effective_data(D_unique, R)
    efficiency = D_eff / D_total * 100
    L = loss_repeated(N, D_unique, R)
    print(f"{R:>10} {fmt_params(D_total):>12} {fmt_params(D_eff):>12} "
          f"{efficiency:>11.1f}% {L:>8.4f}")

# --- 7b: Compare unique vs repeated data ---
print("\n--- Unique vs Repeated Data ---")
print(f"Loss with 4T unique tokens:   {chinchilla_loss(N, 4e12):.4f}")
print(f"Loss with 1T × 4 epochs:      {loss_repeated(N, 1e12, 4):.4f}")
print(f"Loss with 2T × 2 epochs:      {loss_repeated(N, 2e12, 2):.4f}")
print("\n  More unique data always better than repeating!")

# --- 7c: Visualise ---
if HAS_MPL:
    fig, ax = plt.subplots(figsize=(8, 5))
    
    R_range = np.linspace(1, 50, 200)
    
    # Plot effective data multiplier
    for c_val, label in [(0.20, 'c=0.20 (optimistic)'), 
                          (0.28, 'c=0.28 (measured)'), 
                          (0.40, 'c=0.40 (pessimistic)')]:
        D_eff_mult = R_range**(1 - c_val)
        ax.plot(R_range, D_eff_mult, linewidth=2, label=label)
    
    # Perfect scaling (no degradation)
    ax.plot(R_range, R_range, 'k--', linewidth=1, alpha=0.5, label='Perfect (no degradation)')
    
    ax.set_xlabel('Epochs (R)', fontsize=12)
    ax.set_ylabel('Effective Data Multiplier', fontsize=12)
    ax.set_title('Repeated Data: Effective vs Actual', fontsize=14)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('scaling_repeated_data.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: scaling_repeated_data.png")

## §8 MoE Effective Parameters

Mixture-of-Experts: $N_{\text{active}} \ll N_{\text{total}}$.

Effective parameters for scaling comparison:

$$N_{\text{eff}} = N_{\text{active}} \cdot E^\eta, \quad \eta \approx 0.3\text{–}0.5$$

Doubling experts ≈ $2^{0.4} \approx 1.32\times$ effective parameters.

In [ ]:
# === §8: MoE Effective Parameters ===
np.random.seed(42)

def moe_effective_params(N_active, E, eta=0.4):
    """Compute effective parameters for MoE model.
    
    N_eff = N_active * E^eta
    
    Args:
        N_active: active parameters per token
        E: number of experts
        eta: expert efficiency exponent (0 < eta < 1)
    Returns:
        N_eff: effective parameters for scaling comparison
    """
    return N_active * E**eta

def moe_loss(N_active, E, D, eta=0.4, params=CHINCHILLA):
    """Loss for MoE model using effective parameters."""
    N_eff = moe_effective_params(N_active, E, eta)
    return chinchilla_loss(N_eff, D, params)

print("=== MoE Effective Parameters ===\n")

# --- 8a: Effective parameters for various configurations ---
N_active = 7e9  # 7B active per token

print(f"Active parameters: {fmt_params(N_active)}\n")
print(f"{'Experts (E)':>12} {'N_total':>12} {'N_eff':>12} {'Equiv Dense':>12} {'Mem Ratio':>10}")
print("-" * 62)
for E in [1, 2, 4, 8, 16, 32, 64, 128, 256]:
    N_eff = moe_effective_params(N_active, E)
    N_total = N_active * E  # approximate (ignoring shared params)
    mem_ratio = N_total / N_active
    print(f"{E:>12} {fmt_params(N_total):>12} {fmt_params(N_eff):>12} "
          f"{fmt_params(N_eff):>12} {mem_ratio:>10.1f}×")

# --- 8b: MoE vs Dense at same inference cost ---
print("\n--- MoE vs Dense at Same Inference FLOPs ---")
D_train = 2e12  # 2T tokens

dense_N_values = [7e9, 14e9, 20e9, 50e9, 70e9]
moe_E_values = [1, 4, 8, 16, 64]

print(f"\nAll models: {fmt_params(N_active)} active params, {fmt_params(D_train)} training tokens")
print(f"{'Config':>20} {'N_eff':>12} {'Loss':>8} {'PPL':>8}")
print("-" * 52)

# Dense baseline
L_dense = chinchilla_loss(N_active, D_train)
print(f"{'Dense (E=1)':>20} {fmt_params(N_active):>12} {L_dense:>8.4f} {np.exp(L_dense):>8.2f}")

# MoE variants
for E in [4, 8, 16, 64, 256]:
    N_eff = moe_effective_params(N_active, E)
    L_moe = moe_loss(N_active, E, D_train)
    print(f"{'MoE E='+str(E):>20} {fmt_params(N_eff):>12} {L_moe:>8.4f} {np.exp(L_moe):>8.2f}")

# --- 8c: DeepSeek-V3 style analysis ---
print("\n--- DeepSeek-V3 Analysis ---")
N_active_ds = 37e9   # 37B active
E_ds = 256            # ~256 experts
D_ds = 14.8e12        # 14.8T tokens
eta = 0.4

N_eff_ds = moe_effective_params(N_active_ds, E_ds, eta)
L_ds = moe_loss(N_active_ds, E_ds, D_ds, eta)
N_total_ds = 671e9

print(f"DeepSeek-V3:")
print(f"  N_total  = {fmt_params(N_total_ds)}")
print(f"  N_active = {fmt_params(N_active_ds)} per token")
print(f"  E        = {E_ds} experts")
print(f"  N_eff    = {fmt_params(N_eff_ds)} (equivalent dense)")
print(f"  D        = {fmt_params(D_ds)}")
print(f"  L        = {L_ds:.4f}")
print(f"  PPL      = {np.exp(L_ds):.2f}")
print(f"")
print(f"  Inference cost: {N_active_ds/N_eff_ds*100:.0f}% of equivalent dense model")

## §9 Test-Time Compute Scaling

Two mechanisms for test-time scaling:

1. **Best-of-N**: generate $N$ answers, select best → $\text{acc} \approx 1-(1-p)^N$
2. **Chain-of-thought length**: more thinking tokens → $\text{acc} \propto T^\delta$

Improvement scales as $\sqrt{\ln N}$ (extreme value theory).

In [ ]:
# === §9: Test-Time Compute Scaling ===
np.random.seed(42)

def best_of_n_accuracy(p, N):
    """Probability that at least one of N samples is correct.
    
    pass@1 with best-of-N: 1 - (1-p)^N
    """
    return 1 - (1 - p)**N

def extreme_value_improvement(N, sigma=1.0):
    """Expected improvement of best-of-N from extreme value theory.
    
    Best score ~ mu + sigma * sqrt(2 * ln(N))
    Returns improvement in sigma units.
    """
    return sigma * np.sqrt(2 * np.log(N))

def cot_accuracy(T, T_base=100, acc_base=0.30, delta=0.15):
    """Accuracy scaling with chain-of-thought length.
    
    acc(T) = acc_base * (T/T_base)^delta
    """
    return min(acc_base * (T / T_base)**delta, 0.99)

print("=== Test-Time Compute Scaling ===\n")

# --- 9a: Best-of-N ---
print("--- Best-of-N Sampling ---")
base_accuracies = [0.10, 0.20, 0.30, 0.50]
N_values = [1, 2, 4, 8, 16, 32, 64, 128, 256]

print(f"\n{'N':>6}", end='')
for p in base_accuracies:
    print(f"  p={p:.2f}", end='')
print()
print("-" * 40)

for N in N_values:
    print(f"{N:>6}", end='')
    for p in base_accuracies:
        acc = best_of_n_accuracy(p, N)
        print(f"  {acc:>5.1%}", end='')
    print()

# --- 9b: Extreme value improvement ---
print("\n--- Extreme Value Improvement (score improvement in σ units) ---")
for N in [1, 4, 16, 64, 256, 1024]:
    imp = extreme_value_improvement(N)
    print(f"  best-of-{N:>4}: +{imp:.2f}σ  (cost: {N}×)")

# --- 9c: Chain-of-thought length scaling ---
print("\n--- Chain-of-Thought Length Scaling ---")
print(f"Base accuracy: 30% at T=100 tokens, δ=0.15\n")
print(f"{'T (tokens)':>12} {'Accuracy':>10} {'Cost Ratio':>12}")
print("-" * 36)
for T in [100, 200, 400, 800, 1600, 3200, 6400]:
    acc = cot_accuracy(T, delta=0.15)
    cost = T / 100
    print(f"{T:>12} {acc:>10.1%} {cost:>12.0f}×")

# --- 9d: Combined visualisation ---
if HAS_MPL:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    
    # Best-of-N
    N_range = np.arange(1, 129)
    for p, color in zip([0.1, 0.2, 0.3, 0.5], ['red', 'orange', 'green', 'blue']):
        acc = best_of_n_accuracy(p, N_range)
        ax1.plot(N_range, acc * 100, color=color, linewidth=2, label=f'p = {p}')
    
    ax1.set_xlabel('N (samples)', fontsize=12)
    ax1.set_ylabel('Best-of-N Accuracy (%)', fontsize=12)
    ax1.set_title('Best-of-N Sampling', fontsize=14)
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # CoT length scaling
    T_range = np.linspace(100, 6400, 200)
    for delta, color in zip([0.10, 0.15, 0.25], ['blue', 'green', 'red']):
        acc = np.array([cot_accuracy(T, delta=delta) for T in T_range])
        ax2.plot(T_range, acc * 100, color=color, linewidth=2, label=f'δ = {delta}')
    
    ax2.set_xlabel('Thinking Tokens T', fontsize=12)
    ax2.set_ylabel('Accuracy (%)', fontsize=12)
    ax2.set_title('Chain-of-Thought Length Scaling', fontsize=14)
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('scaling_test_time_compute.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: scaling_test_time_compute.png")

## §10 Scaling Law Extrapolation with Uncertainty

Fitted exponents have uncertainty: $\alpha_N = 0.076 \pm 0.01$.

Propagated to loss predictions: wider uncertainty at larger extrapolation distance.

Key tool for decision-making: **confidence intervals on predicted loss**.

In [ ]:
# === §10: Scaling Law Extrapolation with Uncertainty ===
np.random.seed(42)

def loss_with_uncertainty(N, alpha_mean, alpha_std, A, L_ref, N_ref):
    """Compute loss and confidence interval.
    
    L(N) = A * N^(-alpha) calibrated so L(N_ref) = L_ref
    
    Returns: (L_central, L_lower, L_upper)
    """
    # Calibrate A from reference point: L_ref = A * N_ref^(-alpha)
    # → A = L_ref * N_ref^alpha
    
    A_central = L_ref * N_ref**alpha_mean
    L_central = A_central * N**(-alpha_mean)
    
    # Upper/lower bounds (alpha ± std)
    A_low = L_ref * N_ref**(alpha_mean - alpha_std)
    L_upper = A_low * N**(-(alpha_mean - alpha_std))  # smaller alpha → higher loss
    
    A_high = L_ref * N_ref**(alpha_mean + alpha_std)
    L_lower = A_high * N**(-(alpha_mean + alpha_std))  # larger alpha → lower loss
    
    return L_central, L_lower, L_upper

print("=== Scaling Law Extrapolation with Uncertainty ===\n")

# Reference: L(1B) = 3.0, alpha = 0.076 ± 0.01
N_ref = 1e9
L_ref = 3.0
alpha_mean = 0.076
alpha_std = 0.01

print(f"Reference: L({fmt_params(N_ref)}) = {L_ref}")
print(f"Scaling exponent: α = {alpha_mean} ± {alpha_std}\n")

target_Ns = [1e9, 1e10, 1e11, 1e12, 1e13]
print(f"{'N':>10} {'L_central':>10} {'L_lower':>10} {'L_upper':>10} {'Range':>10} {'Rel. Unc.':>10}")
print("-" * 64)

for N in target_Ns:
    L_c, L_lo, L_hi = loss_with_uncertainty(N, alpha_mean, alpha_std, None, L_ref, N_ref)
    range_width = L_hi - L_lo
    rel_unc = range_width / L_c * 100 if L_c > 0 else 0
    print(f"{fmt_params(N):>10} {L_c:>10.4f} {L_lo:>10.4f} {L_hi:>10.4f} "
          f"{range_width:>10.4f} {rel_unc:>9.1f}%")

print("\n  Uncertainty grows with extrapolation distance!")
print(f"  At 1000× extrapolation: ±{(L_hi-L_lo)/L_c*100:.1f}% uncertainty")

# --- 10b: Bootstrap-style uncertainty in Chinchilla N* ---
print("\n--- Uncertainty in Chinchilla-Optimal N* ---")
C = 1e24
print(f"Compute budget: {fmt_sci(C)} FLOPs\n")

# Monte Carlo: sample alpha, beta from distributions
n_samples = 10000
alphas = np.random.normal(0.34, 0.02, n_samples)
betas = np.random.normal(0.28, 0.02, n_samples)

# Compute N* for each sample
N_stars = []
D_stars = []
for al, be in zip(alphas, betas):
    if al > 0 and be > 0:
        params_sample = dict(CHINCHILLA)
        params_sample['alpha'] = al
        params_sample['beta'] = be
        N_opt, D_opt, _ = chinchilla_optimal(C, params_sample)
        N_stars.append(N_opt)
        D_stars.append(D_opt)

N_stars = np.array(N_stars)
D_stars = np.array(D_stars)

print(f"N* distribution (n={len(N_stars)} samples):")
print(f"  Mean:    {fmt_params(np.mean(N_stars))}")
print(f"  Median:  {fmt_params(np.median(N_stars))}")
print(f"  5th %%:   {fmt_params(np.percentile(N_stars, 5))}")
print(f"  95th %%:  {fmt_params(np.percentile(N_stars, 95))}")
print(f"  Range:   {fmt_params(np.percentile(N_stars, 5))} – {fmt_params(np.percentile(N_stars, 95))}")

print(f"\nD* distribution:")
print(f"  Mean:    {fmt_params(np.mean(D_stars))}")
print(f"  5th %%:   {fmt_params(np.percentile(D_stars, 5))}")
print(f"  95th %%:  {fmt_params(np.percentile(D_stars, 95))}")

# --- 10c: Visualise ---
if HAS_MPL:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    
    # Extrapolation fan chart
    N_plot = np.logspace(9, 13, 100)
    L_c_arr, L_lo_arr, L_hi_arr = [], [], []
    for N in N_plot:
        Lc, Llo, Lhi = loss_with_uncertainty(N, alpha_mean, alpha_std, None, L_ref, N_ref)
        L_c_arr.append(Lc)
        L_lo_arr.append(Llo)
        L_hi_arr.append(Lhi)
    
    ax1.fill_between(N_plot, L_lo_arr, L_hi_arr, alpha=0.3, color='blue',
                     label='±1σ confidence band')
    ax1.plot(N_plot, L_c_arr, 'b-', linewidth=2, label='Central estimate')
    ax1.scatter([N_ref], [L_ref], color='red', s=100, zorder=5, label='Reference point')
    ax1.set_xscale('log')
    ax1.set_xlabel('Parameters N', fontsize=12)
    ax1.set_ylabel('Loss L', fontsize=12)
    ax1.set_title('Extrapolation with Uncertainty', fontsize=14)
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # N* distribution histogram
    ax2.hist(N_stars / 1e9, bins=50, color='steelblue', edgecolor='black', alpha=0.7)
    ax2.axvline(np.median(N_stars) / 1e9, color='red', linewidth=2,
                linestyle='--', label=f'Median = {fmt_params(np.median(N_stars))}')
    ax2.set_xlabel('N* (billions)', fontsize=12)
    ax2.set_ylabel('Count', fontsize=12)
    ax2.set_title(f'Distribution of N* at C = {fmt_sci(C)}', fontsize=14)
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('scaling_extrapolation_uncertainty.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: scaling_extrapolation_uncertainty.png")

print("\n" + "=" * 60)
print("THEORY NOTEBOOK COMPLETE")
print("=" * 60)
print("\nKey takeaways:")
print("  1. Power laws: L ∝ N^(-α), L ∝ D^(-β), L ∝ C^(-γ)")
print("  2. Chinchilla: N and D should scale equally (D ≈ 20N)")
print("  3. Inference-optimal: overtrain smaller models for deployment")
print("  4. MoE: N_eff = N_active · E^η; experts help but sublinearly")
print("  5. Test-time compute: best-of-N scales as √ln(N); expensive")
print("  6. Always report uncertainty in scaling law predictions")